In [2]:
import sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LinearRegression
 
OUT = sys.argv[1] if len(sys.argv) > 1 else "out"
PPP = {"India": 20.452242, "United States": 1.0, "Canada": 1.150472, "Singapore": 0.827912,
       "France": 0.681239, "Germany": 0.700862, "Netherlands": 0.731421, "United Kingdom": 0.664153,
       "Switzerland": 0.972889, "Austria": 0.710451, "Spain": 0.562107}
 
g = pd.read_csv('/Users/jahaankachwala/Downloads/Projects /Global Salary Data/skill_salary_clean.csv')
ind = pd.read_csv('/Users/jahaankachwala/Downloads/Projects /Global Salary Data/india_salary_clean.csv')
g = g[g.country_name.isin(PPP)].copy()

In [4]:
# PPP$ = salary in LOCAL currency / PPP factor (local currency per international $)
g["ppp_usd"] = g.salary_mid_local / g.country_name.map(PPP)
g["is_india"] = (g.country_name == "India").astype(int)
 
# sanity: implied market FX per country (local per USD) and currency
chk = g.groupby("country_name").apply(
    lambda d: pd.Series({"currency": d.currency.mode().iloc[0],
                         "market_fx": (d.salary_mid_local / d.salary_mid_usd).median(),
                         "price_level_vs_US": PPP[d.name] / (d.salary_mid_local / d.salary_mid_usd).median()}),
    include_groups=False)
print("=== Sanity: currency, market FX, price level (PPP/market FX; US=1) ===")
print(chk.round(3).to_string())
 
junior = g[g.seniority.isin(["entry", "intern"])]

=== Sanity: currency, market FX, price level (PPP/market FX; US=1) ===
               currency  market_fx  price_level_vs_US
country_name                                         
Austria             EUR      0.890              0.798
Canada              CAD      1.400              0.822
France              EUR      0.890              0.765
Germany             EUR      0.890              0.787
India               INR     85.999              0.238
Netherlands         EUR      0.890              0.822
Singapore           SGD      1.310              0.632
Spain               EUR      0.890              0.632
Switzerland         CHF      0.830              1.172
United Kingdom      GBP      0.760              0.874
United States       USD      1.000              1.000


In [6]:
#1. Nominal vs PPP junior pay
t = junior.groupby("country_name").agg(n=("ppp_usd", "count"),
                                       nominal_usd=("salary_mid_usd", "median"),
                                       ppp_usd=("ppp_usd", "median"))
t = t[t.n >= 20].sort_values("ppp_usd", ascending=False)
us = t.loc["United States"]
t["nominal_%US"] = (t.nominal_usd / us.nominal_usd * 100).round(1)
t["PPP_%US"] = (t.ppp_usd / us.ppp_usd * 100).round(1)
print("\n=== JUNIOR median pay: nominal USD vs PPP-adjusted ===")
print(t.round(0).assign(**{"nominal_%US": t["nominal_%US"], "PPP_%US": t["PPP_%US"]}).to_string())
t.to_csv("junior_pay_ppp.csv")


=== JUNIOR median pay: nominal USD vs PPP-adjusted ===
                   n  nominal_usd  ppp_usd  nominal_%US  PPP_%US
country_name                                                    
Germany           23      53753.0  68259.0        129.2    164.1
Netherlands       54      43483.0  52911.0        104.5    127.2
France            50      37360.0  48808.0         89.8    117.3
Canada           870      37143.0  45199.0         89.3    108.7
United States   6358      41600.0  41600.0        100.0    100.0
United Kingdom    20      36184.0  41406.0         87.0     99.5
Singapore        111      25191.0  39859.0         60.6     95.8
India           1645       3488.0  14668.0          8.4     35.3


In [7]:
#2. Test: India junior vs rest, in PPP$
a = junior[junior.is_india == 1].ppp_usd
b = junior[junior.is_india == 0].ppp_usd
p = stats.mannwhitneyu(a, b, alternative="less")[1]
rng = np.random.default_rng(0)
ratios = [np.median(rng.choice(a, len(a))) / np.median(rng.choice(b, len(b))) for _ in range(1000)]
print(f"\n=== TEST in PPP$: India junior median {a.median():,.0f} vs rest {b.median():,.0f} ===")
print(f"Mann-Whitney one-sided p = {p:.3g}; India/rest ratio 95% CI {np.percentile(ratios,2.5):.2f} - {np.percentile(ratios,97.5):.2f}")


=== TEST in PPP$: India junior median 14,668 vs rest 41,600 ===
Mann-Whitney one-sided p = 0; India/rest ratio 95% CI 0.34 - 0.35


In [8]:
#3. Same role family, PPP, India vs US 
j2 = junior[junior.country_name.isin(["India", "United States"])]
r = j2.pivot_table(index="role_family", columns="country_name", values="ppp_usd", aggfunc=["median", "count"])
r.columns = ["India_med", "US_med", "India_n", "US_n"]
r = r[(r.India_n >= 30) & (r.US_n >= 30)]
r["India_%US_PPP"] = (r.India_med / r.US_med * 100).round(1)
print("\n=== JUNIOR, same role family, PPP$ ===")
print(r.sort_values("India_%US_PPP").round(0).to_string())


=== JUNIOR, same role family, PPP$ ===
                  India_med   US_med  India_n  US_n  India_%US_PPP
role_family                                                       
marketing           11075.0  49920.0      158   201           22.0
software-eng        17113.0  75000.0      256    39           23.0
design              12835.0  52000.0       64    33           25.0
finance             14668.0  58240.0       72   324           25.0
other               13201.0  43680.0      249   884           30.0
hr                  13446.0  44221.0       94   109           30.0
sales               14668.0  45000.0      284   761           33.0
customer-service    13446.0  40000.0      109   271           34.0
admin               14668.0  42640.0       54   185           34.0
manufacturing       14668.0  41600.0       49   671           35.0
healthcare          22002.0  45760.0       39   795           48.0


In [9]:
#4. Seniority, PPP
order = ["intern", "entry", "mid", "senior", "lead", "manager", "director", "executive"]
s = (g[g.country_name.isin(["India", "United States", "Canada"])]
     .pivot_table(index="seniority", columns="country_name", values="ppp_usd", aggfunc="median").reindex(order))
s["India_%US"] = (s.India / s["United States"] * 100).round(1)
print("\n=== MEDIAN PPP$ BY SENIORITY ===")
print(s.round(0).to_string())


=== MEDIAN PPP$ BY SENIORITY ===
country_name    Canada     India  United States  India_%US
seniority                                                 
intern         53891.0    7334.0        41600.0       18.0
entry          45199.0   15891.0        41600.0       38.0
mid                NaN   31781.0        70000.0       45.0
senior         79092.0   46450.0       100000.0       46.0
lead           85180.0   58673.0        80000.0       73.0
manager        67979.0   36671.0        83232.0       44.0
director      106478.0  110012.0       130000.0       85.0
executive     104487.0  207801.0       160000.0      130.0


In [10]:
#5. Regression: India effect after PPP
m = g.dropna(subset=["seniority"]).copy()
m["sen_rank"] = m.seniority.map({k: i for i, k in enumerate(order)})
m["n_skills"] = m.skills.fillna("").apply(lambda x: len(x.split("|")) if x else 0)
X = pd.get_dummies(m[["is_india", "sen_rank", "n_skills", "role_family"]], columns=["role_family"],
                   drop_first=True).astype(float)
y = np.log(m.ppp_usd)
lr = LinearRegression().fit(X, y)
c = dict(zip(X.columns, lr.coef_))
print(f"\n=== REGRESSION log(PPP$) ~ India + seniority + skills + role (n={len(m)}, R2={lr.score(X, y):.2f}) ===")
print(f"India effect: {np.exp(c['is_india'])-1:+.0%} vs other countries, holding seniority/role/skills equal")


=== REGRESSION log(PPP$) ~ India + seniority + skills + role (n=38164, R2=0.42) ===
India effect: -57% vs other countries, holding seniority/role/skills equal


In [11]:
#6. Rs 15 lakh in PPP terms + purchasing-power match 
R15 = 1_500_000
f = PPP["India"]
us_j = junior[junior.country_name == "United States"].ppp_usd.median()
us_all = g[g.country_name == "United States"].ppp_usd.median()
ca_all = g[g.country_name == "Canada"].ppp_usd.median()
print(f"\n=== Rs 15 lakh in PPP$: {R15/f:,.0f} ===")
print(f"US median (all levels): {us_all:,.0f} | US junior median: {us_j:,.0f} | Canada median: {ca_all:,.0f}")
print(f"Rs needed in India to match US junior median purchasing power: Rs {us_j*f/1e5:.1f} lakh")
print(f"Rs needed in India to match US overall median purchasing power: Rs {us_all*f/1e5:.1f} lakh")
print(f"Rs needed in India to match Canada overall median: Rs {ca_all*f/1e5:.1f} lakh")
 
# India experience bands in PPP$ vs US junior median
e = ind.groupby("experience_band").salary_mid_inr.median().reindex(
    ["0-1 yrs", "2-3 yrs", "4-6 yrs", "7-10 yrs", "11-15 yrs", "16+ yrs"])
eb = pd.DataFrame({"median_INR": e, "PPP_usd": (e / f).round(0), "vs_US_junior_%": (e / f / us_j * 100).round(0)})
print("\n=== INDIA by experience (PPP$) vs US junior median PPP$ ===")
print(eb.to_string())
print(f"Share of Indian postings at or above Rs {us_j*f/1e5:.1f} lakh (US-junior purchasing power): "
      f"{(ind.salary_mid_inr >= us_j*f).mean()*100:.1f}%")
print(f"Share at or above Rs {us_all*f/1e5:.1f} lakh (US-median purchasing power): "
      f"{(ind.salary_mid_inr >= us_all*f).mean()*100:.1f}%")


=== Rs 15 lakh in PPP$: 73,342 ===
US median (all levels): 60,000 | US junior median: 41,600 | Canada median: 54,326
Rs needed in India to match US junior median purchasing power: Rs 8.5 lakh
Rs needed in India to match US overall median purchasing power: Rs 12.3 lakh
Rs needed in India to match Canada overall median: Rs 11.1 lakh

=== INDIA by experience (PPP$) vs US junior median PPP$ ===
                 median_INR   PPP_usd  vs_US_junior_%
experience_band                                      
0-1 yrs            300000.0   14668.0            35.0
2-3 yrs            525000.0   25670.0            62.0
4-6 yrs            850000.0   41560.0           100.0
7-10 yrs          1300000.0   63563.0           153.0
11-15 yrs         1800000.0   88010.0           212.0
16+ yrs           2125000.0  103901.0           250.0
Share of Indian postings at or above Rs 8.5 lakh (US-junior purchasing power): 33.5%
Share at or above Rs 12.3 lakh (US-median purchasing power): 23.9%


In [15]:
from pathlib import Path

#Please note that the code above is a continuation of the previous code snippet and includes additional analysis and calculations related to salary data, particularly focusing on India and its comparison with other countries in terms of purchasing power parity (PPP) and salary distributions.
top = t.head(9)
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(len(top)); w = 0.4
ax[0].bar(x - w/2, top.nominal_usd, w, label="Nominal USD")
ax[0].bar(x + w/2, top.ppp_usd, w, label="PPP-adjusted")
ax[0].set_xticks(x); ax[0].set_xticklabels(top.index, rotation=40, ha="right")
ax[0].set_title("Junior median pay: nominal vs PPP"); ax[0].legend()
eb.PPP_usd.plot.bar(ax=ax[1], color="#55a868")
ax[1].axhline(us_j, color="red", ls="--", label=f"US junior median (PPP$ {us_j:,.0f})")
ax[1].set_title("India: median pay by experience (PPP$)"); ax[1].legend(); ax[1].set_xlabel("")
OUT = Path("out")
OUT.mkdir(parents=True, exist_ok=True)
plt.tight_layout(); plt.savefig(f"{OUT}/ppp_analysis.png", dpi=130)
print("saved")

saved
